In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [3]:
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'ratings': [1, 0, 1, 0, 1, 0] # 1: 긍정, 0: 부정
}

In [4]:
import pandas as pd
df = pd.DataFrame(data)
df.head(1)

,reviews,ratings
0,이 영화 정말 좋아 최고야,1


In [7]:
# 전처리
import re
def clean_data(text):
    return re.sub(r'[^가-힣\s]', '', text)
df['clean'] = df['reviews'].apply(lambda x : clean_data(x))

# 토큰분류 형태소
from konlpy.tag import Okt
okt = Okt()
def kor_tokened(text):    
    return [word for word, pos in okt.pos(text,stem=True) if len(word) > 1]

df['tokens'] = df['clean'].apply(lambda x : kor_tokened(x))

In [20]:
df['tokens']

0        [영화, 정말, 좋다, 최고]
1      [시간, 아깝다, 쓰레기, 영화]
2      [배우, 연기, 너무, 훌륭하다]
3        [스토리, 지루하다, 뻔하다]
4    [인생, 최고, 명작, 이다, 추천]
5       [주다, 보기, 아깝다, 졸작]
Name: tokens, dtype: object

In [21]:
# 단어인덱싱
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
def make_vocab(tokens):
    for token in tokens:    
        if token not in vocab:
            vocab[token] = len(vocab)
df['tokens'].apply(lambda x : make_vocab(x))

0    None
1    None
2    None
3    None
4    None
5    None
Name: tokens, dtype: object

In [37]:
# padding  길이 맞추기 
MAX_LEN = 5
def to_sequence(tokens):
    # 토큰화 - 패딩    
    x = [vocab.get(word,1) for word in tokens ]
    if len(x) < MAX_LEN:
        x = x + [0]*(MAX_LEN-len(x))
    else:
        x = x[:MAX_LEN]
    return x

df['pad_tokens'] =  df['tokens'].apply(lambda x : to_sequence(x))
df

,reviews,ratings,clean,tokens,pad_tokens
0,이 영화 정말 좋아 최고야,1,이 영화 정말 좋아 최고야,"[영화, 정말, 좋다, 최고]","[2, 3, 4, 5, 0]"
1,시간 아까운 쓰레기 영화,0,시간 아까운 쓰레기 영화,"[시간, 아깝다, 쓰레기, 영화]","[6, 7, 8, 2, 0]"
2,배우들 연기가 너무 훌륭해요,1,배우들 연기가 너무 훌륭해요,"[배우, 연기, 너무, 훌륭하다]","[9, 10, 11, 12, 0]"
3,스토리가 지루하고 뻔하다,0,스토리가 지루하고 뻔하다,"[스토리, 지루하다, 뻔하다]","[13, 14, 15, 0, 0]"
4,인생 최고의 명작입니다 추천,1,인생 최고의 명작입니다 추천,"[인생, 최고, 명작, 이다, 추천]","[16, 5, 17, 18, 19]"
5,돈 주고 보기 아까운 졸작,0,돈 주고 보기 아까운 졸작,"[주다, 보기, 아깝다, 졸작]","[20, 21, 7, 22, 0]"


In [39]:
# 데이터셋, -- > Tensor타입변경 
# 데이터로더 --> 배치단위로 학습
from torch.utils.data  import Dataset, DataLoader
class ReviewMovieDataset(Dataset):
    def __init__(self,x,y):
        self.x = torch.LongTensor(x)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, index):
        self.x[index], self.y[index]
x = df['pad_tokens'].tolist();  y = df['ratings'].tolist()
dataset = ReviewMovieDataset(x,y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)        